# A seleção que mais marcou em cada Copa

## Task 3: a seleção que mais marcou em cada Copa

## A pergunta do time de analytics

**Qual seleção mais marcou gols em cada Copa?**

In [ ]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-rankings")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"

spark

A célula abaixo já vem pronta: o código dos vídeos 2.4 e 3.1, na ordem, com as cinco views do projeto.

In [ ]:
# vídeo 2.4: escalações e Copas
from wc_pipeline.schemas import PLAYERS, WORLD_CUPS

escalacoes = (
    spark.read.option("header", "true").schema(PLAYERS).csv(f"{RAW}/WorldCupPlayers.csv")
    .select(
        F.col("MatchID").alias("match_id"),
        F.col("Team Initials").alias("sigla"),
        F.col("Player Name").alias("jogador"),
        F.col("Position").alias("posicao"),
    )
)

copas = (
    spark.read.option("header", "true").schema(WORLD_CUPS).csv(f"{RAW}/WorldCups.csv")
    .select(
        F.col("Year").alias("ano"),
        F.col("Country").alias("sede"),
        F.col("Winner").alias("campeao"),
    )
)

# vídeo 3.1: as tabelas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas, partidas_por_selecao, resumo_por_selecao

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)
por_selecao = partidas_por_selecao(partidas)
resumo = resumo_por_selecao(por_selecao)

# vídeo 3.1: as views
partidas.createOrReplaceTempView("partidas")
por_selecao.createOrReplaceTempView("por_selecao")
resumo.createOrReplaceTempView("resumo")
escalacoes.createOrReplaceTempView("escalacoes")
copas.createOrReplaceTempView("copas")

Uma parte do resumo, também registrada como view:

## Por que o groupBy não basta

O `groupBy` junta as linhas de cada Copa numa só, e a seleção se perde. Uma **window function** calcula olhando o
grupo, mas **mantém todas as linhas**, com o resultado numa coluna nova.

| Parte | Pergunta | No nosso caso |
|---|---|---|
| `partitionBy` | quais linhas formam um grupo? | as seleções de uma **mesma Copa** |
| `orderBy` | em que ordem, dentro do grupo? | dos **mais gols** para os menos |

## A posição de cada seleção na sua Copa

`row_number()` numera as linhas de cada grupo: 1, 2, 3... O saldo e o nome entram no `orderBy` como desempate.

A posição 1 de cada Copa:

## A mesma pergunta em SQL, com `QUALIFY`

`OVER (PARTITION BY ... ORDER BY ...)` é a janela; o `QUALIFY` filtra pelo resultado dela.

## Para consultar depois

- **`rank` e `dense_rank`:** também numeram as linhas, mas tratam empates de outro jeito. Com dois empatados em
  segundo, `rank` dá 1, 2, 2, 4 (pula a posição) e `dense_rank` dá 1, 2, 2, 3. O `row_number` sempre dá 1, 2, 3, 4.
- **`lag` e `lead`:** pegam o valor da linha anterior ou da próxima dentro do grupo, por exemplo os gols da seleção
  na Copa anterior (janela por seleção, ordenada por ano).
- **Acumulado:** `F.sum(...)` sobre uma janela com `rowsBetween(Window.unboundedPreceding, Window.currentRow)` soma do
  início do grupo até a linha atual.
- **Janela sem `orderBy`** vê o grupo inteiro: serve para colocar o total do grupo em cada linha e calcular
  percentuais.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()